In [3]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

Load prepared data

In [4]:
df_sentiment_path = "./savedData/df_sentiment.csv"

if os.path.exists(df_sentiment_path):
    df_sentiment = pd.read_csv(df_sentiment_path)
    df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'])
    print("Already Saved")
else:
    print("Start the data_loader to write file into savedData")

Already Saved


In [10]:
df_sentiment.head()

,Date,text,Sentiment,sentiment_score,avg_sentiment,next_day_price
0,2019-05-27,È appena uscito un nuovo video! LES CRYPTOMONN...,Positive,1,1.0,8719.961914
1,2019-05-27,Cardano: Digitize Currencies; EOS https://t.co...,Positive,1,1.0,8719.961914
2,2019-05-27,Another Test tweet that wasn't caught in the s...,Positive,1,1.0,8719.961914
3,2019-05-27,Current Crypto Prices! \n\nBTC: $8721.99 USD\n...,Positive,1,1.0,8719.961914
4,2019-05-27,Spiv (Nosar Baz): BITCOIN Is An Asset &amp; NO...,Positive,1,1.0,8719.961914


In [11]:
df_sentiment.dtypes

Date               datetime64[ns]
text                       object
Sentiment                  object
sentiment_score             int64
avg_sentiment             float64
next_day_price            float64
dtype: object

Finbert laden

In [12]:
tokenizer = AutoTokenizer.from_pretrained('yiyanghkust/finbert-pretrain')
model = AutoModel.from_pretrained('yiyanghkust/finbert-pretrain')

In [13]:
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30873, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [14]:
def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()  # [CLS]-Token

In [ ]:
embeddings = np.stack([get_bert_embedding(t) for t in df_sentiment['text']])

# Combine embeddings, sentiment score and avg sentiment per day
X = np.hstack([embeddings,
               df_sentiment[['sentiment_score', 'avg_sentiment']].values])
# For now just the next day price
y = df_sentiment['next_day_price'].values

In [ ]:
# Zusammenführen zum Speichern
Xy_stack = np.hstack([X, y.reshape(-1, 1)])
Xy_df = pd.DataFrame(Xy_stack)
Xy_df.to_csv(XY_path, index=False)

Checks if the embeddings already been done.
If not it has to calculate them again.
The calculation takes a very long time especially when it´s performed on the cpu

In [ ]:
XY_path = "./savedData/X_y.csv"

if os.path.exists(XY_path):
    # Daten laden
    data = pd.read_csv(XY_path)
    # Features und Ziel trennen (hier als Beispiel Annahme, dass 'y' die letzte Spalte ist)
    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].values
    print("Features und Ziel aus Datei geladen.")
else:
    # Embeddings neu berechnen
    embeddings = np.stack([get_bert_embedding(t) for t in df_sentiment['text']])
    # Features zusammenbauen
    X = np.hstack([embeddings,
                   df_sentiment[['sentiment_score', 'avg_sentiment']].values])
    y = df_sentiment['next_day_price'].values
    
    # Zusammenführen zum Speichern
    Xy_stack = np.hstack([X, y.reshape(-1, 1)])
    Xy_df = pd.DataFrame(Xy_stack)
    Xy_df.to_csv(XY_path, index=False)
    print("Features und Ziel neu berechnet und gespeichert.")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = RandomForestRegressor()
model.fit(X_train, y_train)
score = model.score(X_test, y_test)
print("R^2 auf Testdaten:", score)